I'm solving the "Blackbox AI 1.0" challenge. My black box is system GK-06, a synthetic cargo inspection / shipment approval service: given one shipment record it returns `score` (0.0-1.0) and `decision` (APPROVE/DECLINE). It is deterministic (identical inputs always give identical outputs). The feature names are only labels, so trust the data, not real-world cargo/security intuition.

DATA: I have a CSV file containing my 74 genuine logged black-box queries. The file is named `GK06_dataset.csv`.

Columns are:

query_key
round
query_id
container_count
declared_value
discrepancy_ratio
port
prior_shipments
recent_seizures
route_age_days
shipper_score
shipper_years
transfers
score
decision

The dataset contains 56 Round-1 queries and 18 Round-2 queries. The observations were generated mostly by changing one feature at a time from a previous query, so the data is NOT i.i.d.

Valid input ranges:

container_count: 0-100
declared_value: 0-100
discrepancy_ratio: 0-1
port: A, B, C, D
prior_shipments: 0-20
recent_seizures: 0-5
route_age_days: 18-75
shipper_score: 300-900
shipper_years: 0-40
transfers: 0-6

GOAL:

Do NOT build an LLM.

Build a surrogate model that reproduces the GK-06 black box as accurately as possible: predict `score`, then derive `decision`.

Use ONLY the genuine observations in `GK06_dataset.csv`. Do NOT fabricate or synthetically generate additional training observations.

Write complete, runnable Google Colab code in separate cells with short comments.

The notebook must be designed specifically for GK-06 and must not use the GK-02 hospital features.

STEPS:

1. LOAD AND CLEAN DATA

- Upload/read `GK06_dataset.csv`.
- Inspect shape, columns, dtypes and missing values.
- Parse `round` and `query_id`.
- Sort chronologically by round and query_id.
- Display `info()` and `describe()`.
- Define the GK-06 feature list.
- One-hot encode `port`.
- Separate X features and y=`score`.
- Preserve the original `decision` column for validation.

2. SENSITIVITY ANALYSIS

Because many queries differ from the previous query in exactly one feature:

- Compare consecutive chronological rows.
- Find pairs where exactly one input feature changed.
- For every such pair calculate:
    feature changed
    old value
    new value
    old score
    new score
    score change
    absolute score change
- Group by feature.
- Calculate mean absolute score change, median absolute score change, maximum absolute score change and number of observations.
- Rank features by observed influence.
- Identify features that appear irrelevant or weakly influential based ONLY on the observed data.
- Do not assume any feature is irrelevant before calculating this.

3. VISUAL ANALYSIS

Create:

- score vs container_count
- score vs declared_value
- score vs discrepancy_ratio
- score vs prior_shipments
- score vs recent_seizures
- score vs route_age_days
- score vs shipper_score
- score vs shipper_years
- score vs transfers
- score by port
- correlation heatmap
- score histogram separated by APPROVE/DECLINE

Use readable labels and titles.

4. DECISION THRESHOLD

Determine the decision rule from the observed data.

Find the exact score threshold separating APPROVE and DECLINE.

Check:

predicted_decision = APPROVE if score >= threshold
otherwise DECLINE

Then compare this derived decision against every observed decision.

Report:

- threshold
- number correctly classified
- number incorrectly classified
- accuracy

If a single threshold cannot perfectly reproduce all observed decisions, explicitly report that instead of forcing one.

5. MODEL TRAINING AND COMPARISON

Train surrogate score models using:

A. LightGBM regressor
B. Explainable Boosting Machine regressor from `interpret`
C. GaussianProcessRegressor with a Matern kernel and scaled inputs
D. Small MLPRegressor
E. Symbolic/additive baseline using a simple interpretable additive model such as Ridge regression

Use appropriate preprocessing.

For LightGBM, only use monotonic constraints if the observed data provides strong evidence for a monotonic relationship. Do NOT impose monotonicity merely from domain intuition.

Evaluate models using BOTH:

A. Repeated KFold cross-validation
B. GroupKFold with `round` as the group

Report:

Score prediction:
- RMSE
- MAE
- R²

Decision prediction:
- Accuracy
- F1 score

Decision predictions must be derived from the model's predicted score using the threshold discovered from the real black-box observations.

Because there are only 74 observations and only 2 rounds, clearly warn about overfitting and the limitations of GroupKFold.

6. MODEL EXPLANATION

Explain the strongest surrogate model.

Use SHAP where practical, otherwise use EBM explanations.

Generate:

- global feature importance
- feature effect plots where supported
- explanation of which variables most affect predicted score

Also investigate these interactions:

- route_age_days × shipper_score
- discrepancy_ratio × shipper_score
- route_age_days × discrepancy_ratio

Do NOT assume these interactions exist. Test them using the data.

7. ACTIVE LEARNING

Use the trained Gaussian Process uncertainty (`std`) or ensemble variance.

Generate candidate records within the valid GK-06 ranges.

Find the 20 most informative NEXT queries.

The candidates should prioritize:

- predictions near the decision boundary
- high model uncertainty
- regions of the valid feature space that are poorly represented in the current 74 observations
- potentially informative feature interactions

Return a table containing EVERY input field:

container_count
declared_value
discrepancy_ratio
port
prior_shipments
recent_seizures
route_age_days
shipper_score
shipper_years
transfers

Also include:

predicted_score
predicted_decision
uncertainty

These must be valid rows that I can manually enter into the black-box query console.

IMPORTANT:
Do not claim these are known black-box outputs. They are only model-based candidate queries for future experimentation.

8. FINAL SURROGATE API

Create:

predict(record: dict) -> {"score": float, "decision": str}

Requirements:

- accept one GK-06 record as a Python dictionary
- preprocess it exactly like training data
- predict score
- clip score to [0,1]
- derive decision using the discovered threshold
- return a dictionary

Also implement:

retrain(new_rows_df)

This function must:

- append genuinely observed new black-box rows
- update the stored training dataset
- refit the surrogate model
- preserve the same preprocessing pipeline
- update the model used by `predict()`

Include a test example using one of the existing GK-06 rows.

9. FINAL PLAIN-ENGLISH SUMMARY

At the end, automatically summarize what the data actually supports.

Discuss:

- strongest observed drivers
- weakest/possibly irrelevant features
- important nonlinear effects
- important interactions
- port effects
- approximate decision threshold
- best surrogate model
- uncertainty/limitations caused by only 74 observations
- which 20 future queries would be most informative

Do NOT invent conclusions.

Do NOT use the GK-02 hospital hypotheses.

Do NOT fabricate data.

Install anything needed:

pip install lightgbm interpret shap

Avoid deprecated APIs.

Make the notebook runnable in Google Colab from top to bottom.

Use separate numbered cells with short comments.

The final result should be a complete GK-06 black-box reverse-engineering notebook based on my actual 74 observations.



In [ ]:
!pip install -q lightgbm interpret shap
print("Libraries successfully installed.")

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())


In [ ]:
# Parse round/query information

if "round" in df.columns:
    df["round_num"] = (
        df["round"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(int)
    )

if "query_id" in df.columns:
    df["query_num"] = df["query_id"].astype(int)

# Sort chronologically
df = df.sort_values(
    ["round_num", "query_num"]
).reset_index(drop=True)

print("Dataset sorted chronologically.")
display(df.head())

In [ ]:
print("=== DATASET INFORMATION ===")
df.info()

print("\n=== DESCRIPTIVE STATISTICS ===")
display(df.describe(include="all"))

In [ ]:
feature_cols = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "port",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers"
]

numeric_features = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers"
]

print("GK-06 input features:")
for i, f in enumerate(feature_cols, 1):
    print(f"{i}. {f}")

In [ ]:
feature_cols = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "port",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers"
]

numeric_features = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers"
]

print("GK-06 input features:")
for i, f in enumerate(feature_cols, 1):
    print(f"{i}. {f}")

In [ ]:
feature_cols = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "port",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers"
]

numeric_features = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers"
]

print("GK-06 input features:")
for i, f in enumerate(feature_cols, 1):
    print(f"{i}. {f}")

In [ ]:
# One-hot encode the categorical port feature

model_df = pd.get_dummies(
    df,
    columns=["port"],
    prefix="port",
    dtype=float
)

port_cols = [
    "port_A",
    "port_B",
    "port_C",
    "port_D"
]

model_features = numeric_features + port_cols

X = model_df[model_features].astype(float)
y = model_df["score"].astype(float)

y_decision = (
    model_df["decision"] == "APPROVE"
).astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
print("=== DECISION DISTRIBUTION ===")
print(df["decision"].value_counts())

print("\n=== SCORE RANGE ===")
print("Minimum score:", df["score"].min())
print("Maximum score:", df["score"].max())

In [ ]:
print("Executing GK-06 Sensitivity Analysis...")

consecutive_diffs = []

for i in range(len(df) - 1):

    row1 = df.iloc[i]
    row2 = df.iloc[i + 1]

    diffs = {}

    # Numeric features
    for col in numeric_features:

        if row1[col] != row2[col]:

            diffs[col] = {
                "feature_change": row2[col] - row1[col],
                "score_change": row2["score"] - row1["score"]
            }

    # Port
    if row1["port"] != row2["port"]:

        diffs["port"] = {
            "feature_change":
                f"{row1['port']} -> {row2['port']}",
            "score_change":
                row2["score"] - row1["score"]
        }

    # Keep only cases where exactly ONE feature changed
    if len(diffs) == 1:

        feature = list(diffs.keys())[0]

        consecutive_diffs.append({
            "Feature": feature,
            "Delta_Feature":
                diffs[feature]["feature_change"],
            "Delta_Score":
                diffs[feature]["score_change"],
            "Abs_Delta_Score":
                abs(diffs[feature]["score_change"])
        })

diff_df = pd.DataFrame(consecutive_diffs)

if len(diff_df) > 0:

    rankings = (
        diff_df
        .groupby("Feature")["Abs_Delta_Score"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    print("\n=== FEATURE INFLUENCE RANKING ===")
    display(rankings)

else:

    print("No clean one-feature mutations found.")

In [ ]:
print("=== INDIVIDUAL ONE-FEATURE CHANGES ===")

if len(diff_df) > 0:
    display(
        diff_df.sort_values(
            "Abs_Delta_Score",
            ascending=False
        )
    )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for feature in numeric_features:

    plt.figure(figsize=(7, 4))

    sns.scatterplot(
        data=df,
        x=feature,
        y="score",
        hue="decision"
    )

    plt.title(f"GK-06 Score vs {feature}")
    plt.xlabel(feature)
    plt.ylabel("Black-box score")

    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="port",
    y="score"
)

plt.title("GK-06 Score by Port")
plt.xlabel("Port")
plt.ylabel("Score")

plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

corr_features = numeric_features + ["score"]

sns.heatmap(
    df[corr_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("GK-06 Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="score",
    hue="decision",
    kde=True,
    bins=25
)

plt.title("GK-06 Score Distribution by Decision")
plt.xlabel("Score")
plt.ylabel("Number of Queries")

plt.show()

In [ ]:
decline_scores = df[
    df["decision"] == "DECLINE"
]["score"]

approve_scores = df[
    df["decision"] == "APPROVE"
]["score"]

max_decline = decline_scores.max()
min_approve = approve_scores.min()

threshold = (
    max_decline + min_approve
) / 2

print("Maximum DECLINE score:", max_decline)
print("Minimum APPROVE score:", min_approve)
print("Estimated decision threshold:", threshold)

In [ ]:
threshold_predictions = np.where(
    df["score"] >= threshold,
    "APPROVE",
    "DECLINE"
)

correct = (
    threshold_predictions ==
    df["decision"].values
)

print(
    f"Correct classifications: "
    f"{correct.sum()} / {len(correct)}"
)

print(
    "Threshold perfectly reproduces observed decisions:",
    correct.all()
)

In [ ]:
from sklearn.model_selection import (
    GroupKFold,
    RepeatedKFold
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    f1_score
)

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)

from sklearn.gaussian_process.kernels import (
    Matern
)

from sklearn.neural_network import (
    MLPRegressor
)

from sklearn.linear_model import Ridge

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

import lightgbm as lgb

from interpret.glassbox import (
    ExplainableBoostingRegressor
)

import warnings

warnings.filterwarnings("ignore")

print("ML libraries loaded.")

In [ ]:
models = {

    "LightGBM":
        lgb.LGBMRegressor(
            n_estimators=50,
            max_depth=3,
            learning_rate=0.05,
            random_state=42,
            verbose=-1
        ),

    "EBM":
        ExplainableBoostingRegressor(
            random_state=42
        ),

    "GPR":
        make_pipeline(
            StandardScaler(),
            GaussianProcessRegressor(
                kernel=Matern(nu=1.5),
                alpha=0.01,
                random_state=42
            )
        ),

    "MLP":
        make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(32, 16),
                max_iter=1000,
                early_stopping=True,
                random_state=42
            )
        ),

    "Ridge":
        make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0)
        )
}

print("Models prepared:")
for name in models:
    print("-", name)

In [ ]:
models = {

    "LightGBM":
        lgb.LGBMRegressor(
            n_estimators=50,
            max_depth=3,
            learning_rate=0.05,
            random_state=42,
            verbose=-1
        ),

    "EBM":
        ExplainableBoostingRegressor(
            random_state=42
        ),

    "GPR":
        make_pipeline(
            StandardScaler(),
            GaussianProcessRegressor(
                kernel=Matern(nu=1.5),
                alpha=0.01,
                random_state=42
            )
        ),

    "MLP":
        make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(32, 16),
                max_iter=1000,
                early_stopping=True,
                random_state=42
            )
        ),

    "Ridge":
        make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0)
        )
}

print("Models prepared:")
for name in models:
    print("-", name)

In [ ]:
rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

results = {}

for name in models:

    results[name] = {
        "rmse": [],
        "mae": [],
        "r2": [],
        "accuracy": [],
        "f1": []
    }

for train_idx, test_idx in rkf.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    decision_test = y_decision.iloc[test_idx]

    for name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        predictions = model.predict(
            X_test
        )

        predictions = np.clip(
            predictions,
            0,
            1
        )

        decision_predictions = (
            predictions >= threshold
        ).astype(int)

        results[name]["rmse"].append(
            np.sqrt(
                mean_squared_error(
                    y_test,
                    predictions
                )
            )
        )

        results[name]["mae"].append(
            mean_absolute_error(
                y_test,
                predictions
            )
        )

        results[name]["r2"].append(
            r2_score(
                y_test,
                predictions
            )
        )

        results[name]["accuracy"].append(
            accuracy_score(
                decision_test,
                decision_predictions
            )
        )

        results[name]["f1"].append(
            f1_score(
                decision_test,
                decision_predictions,
                zero_division=0
            )
        )

print("Cross-validation complete.")

In [ ]:
performance = []

for name in results:

    performance.append({

        "Model": name,

        "RMSE":
            np.mean(
                results[name]["rmse"]
            ),

        "MAE":
            np.mean(
                results[name]["mae"]
            ),

        "R2":
            np.mean(
                results[name]["r2"]
            ),

        "Decision Accuracy":
            np.mean(
                results[name]["accuracy"]
            ),

        "F1":
            np.mean(
                results[name]["f1"]
            )
    })

performance_df = pd.DataFrame(
    performance
).sort_values("RMSE")

display(performance_df)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    random_state=42
)

rf.fit(X, y)

importance = pd.Series(
    rf.feature_importances_,
    index=model_features
).sort_values(
    ascending=False
)

print("=== GK-06 FEATURE IMPORTANCE ===")

display(
    importance.to_frame(
        "importance"
    )
)


In [ ]:
plt.figure(figsize=(10, 7))

importance.sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("GK-06 Random Forest Feature Importance")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="route_age_days",
    y="score",
    hue="shipper_score",
    size="discrepancy_ratio",
    palette="viridis"
)

plt.title(
    "GK-06 Interaction: Route Age × Shipper Score"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="discrepancy_ratio",
    y="score",
    hue="shipper_score",
    size="recent_seizures",
    palette="plasma"
)

plt.title(
    "GK-06 Interaction: Discrepancy Ratio × Shipper Score"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="route_age_days",
    y="score",
    hue="discrepancy_ratio",
    palette="coolwarm"
)

plt.title(
    "GK-06 Interaction: Route Age × Discrepancy Ratio"
)

plt.show()


In [ ]:
final_scaler = StandardScaler()

X_scaled = final_scaler.fit_transform(X)

final_gp = GaussianProcessRegressor(
    kernel=Matern(nu=1.5),
    alpha=0.01,
    random_state=42
)

final_gp.fit(
    X_scaled,
    y
)

print("Final GPR surrogate trained.")

In [ ]:
np.random.seed(100)

n_candidates = 5000

candidates = pd.DataFrame({

    "container_count":
        np.random.randint(
            0, 101, n_candidates
        ),

    "declared_value":
        np.random.randint(
            0, 101, n_candidates
        ),

    "discrepancy_ratio":
        np.round(
            np.random.uniform(
                0, 1, n_candidates
            ),
            2
        ),

    "prior_shipments":
        np.random.randint(
            0, 21, n_candidates
        ),

    "recent_seizures":
        np.random.randint(
            0, 6, n_candidates
        ),

    "route_age_days":
        np.random.randint(
            18, 76, n_candidates
        ),

    "shipper_score":
        np.random.randint(
            300, 901, n_candidates
        ),

    "shipper_years":
        np.random.randint(
            0, 41, n_candidates
        ),

    "transfers":
        np.random.randint(
            0, 7, n_candidates
        )
})

# Random valid ports
candidate_ports = np.random.choice(
    ["A", "B", "C", "D"],
    n_candidates
)

for port in ["A", "B", "C", "D"]:

    candidates[
        f"port_{port}"
    ] = (
        candidate_ports == port
    ).astype(float)

In [ ]:
candidate_X = final_scaler.transform(
    candidates[model_features]
)

pred_mean, pred_std = final_gp.predict(
    candidate_X,
    return_std=True
)

distance_to_boundary = np.abs(
    pred_mean - threshold
)

# High uncertainty + close to decision boundary
active_score = (
    pred_std /
    (distance_to_boundary + 0.05)
)

candidates["predicted_score"] = pred_mean
candidates["uncertainty"] = pred_std
candidates["active_score"] = active_score

proposals = (
    candidates
    .sort_values(
        "active_score",
        ascending=False
    )
    .head(20)
    .copy()
)

print(
    "=== TOP 20 INFORMATIVE NEXT QUERIES ==="
)

display(
    proposals[
        [
            "container_count",
            "declared_value",
            "discrepancy_ratio",
            "prior_shipments",
            "recent_seizures",
            "route_age_days",
            "shipper_score",
            "shipper_years",
            "transfers",
            "predicted_score",
            "uncertainty"
        ]
    ]
)

In [ ]:
proposals["port"] = np.select(
    [
        proposals["port_A"] == 1,
        proposals["port_B"] == 1,
        proposals["port_C"] == 1,
        proposals["port_D"] == 1
    ],
    [
        "A",
        "B",
        "C",
        "D"
    ],
    default="A"
)

query_columns = [
    "container_count",
    "declared_value",
    "discrepancy_ratio",
    "port",
    "prior_shipments",
    "recent_seizures",
    "route_age_days",
    "shipper_score",
    "shipper_years",
    "transfers",
    "predicted_score",
    "uncertainty"
]

display(
    proposals[
        query_columns
    ].reset_index(drop=True)
)

In [ ]:
class GK06Surrogate:

    def __init__(self):

        self.feature_names = model_features

        self.scaler = StandardScaler()

        self.model = GaussianProcessRegressor(
            kernel=Matern(nu=1.5),
            alpha=0.01,
            random_state=42
        )

        self.threshold = threshold

    def fit(self, training_df):

        temp = pd.get_dummies(
            training_df,
            columns=["port"],
            dtype=float
        )

        X_train = temp[
            self.feature_names
        ].astype(float)

        y_train = temp["score"].astype(float)

        self.scaler.fit(X_train)

        X_scaled = self.scaler.transform(
            X_train
        )

        self.model.fit(
            X_scaled,
            y_train
        )

        decline = training_df[
            training_df["decision"] == "DECLINE"
        ]["score"]

        approve = training_df[
            training_df["decision"] == "APPROVE"
        ]["score"]

        if len(decline) and len(approve):

            self.threshold = (
                decline.max() +
                approve.min()
            ) / 2

    def predict(self, record):

        row = pd.DataFrame([record])

        row = pd.get_dummies(
            row,
            columns=["port"],
            dtype=float
        )

        for col in self.feature_names:

            if col not in row.columns:

                row[col] = 0.0

        X_row = row[
            self.feature_names
        ].astype(float)

        X_scaled = self.scaler.transform(
            X_row
        )

        score = float(
            self.model.predict(
                X_scaled
            )[0]
        )

        score = np.clip(
            score,
            0,
            1
        )

        decision = (
            "APPROVE"
            if score >= self.threshold
            else "DECLINE"
        )

        return {
            "score": round(
                float(score),
                4
            ),
            "decision": decision
        }

    def retrain(self, new_rows_df):

        combined = pd.concat(
            [df, new_rows_df],
            ignore_index=True
        )

        self.fit(combined)

In [ ]:
surrogate = GK06Surrogate()

surrogate.fit(df)

test_record = {
    "container_count": 99,
    "declared_value": 4,
    "discrepancy_ratio": 0.9,
    "port": "D",
    "prior_shipments": 0,
    "recent_seizures": 4,
    "route_age_days": 30,
    "shipper_score": 900,
    "shipper_years": 30,
    "transfers": 4
}

print("Test input:")
print(test_record)

print("\nSurrogate prediction:")
print(
    surrogate.predict(
        test_record
    )
)

In [ ]:
output_path = "/content/GK06_analysis_dataset.csv"

df.to_csv(
    output_path,
    index=False
)

print(
    "Saved:",
    output_path
)